#Group 16:
#Nikhil Aji
##&
#Mohammad Aljabrery

## Task 1
## Request to retrieve video ID, title, publish time, view count, and comment count for all videos from the 3Blue1Brown channel:

In [0]:
%pip install google-api-python-client

In [0]:
import getpass
import pandas as pd
from googleapiclient.discovery import build

API_KEY = getpass.getpass('Please enter your API key below then press ENTER: ')
CHANNEL_ID = "UCYO_jab_esuFRV4b17AJtAw"   # this is the channel id, the name is 3Blue1Brown

youtube = build("youtube", "v3", developerKey=API_KEY)

## Getting the total number of videos in his channel:

In [0]:
def get_channel_video_count(api_key, channel_id):
    youtube = build("youtube", "v3", developerKey=api_key)

    request = youtube.channels().list(
        part="statistics,snippet",
        id=channel_id
    )
    response = request.execute()

    if response["items"]:
        stats = response["items"][0]["statistics"]
        channel_title = response["items"][0]["snippet"]["title"]
        video_count = int(stats.get("videoCount", 0))

        print(f"Channel: {channel_title}")
        print(f"Video Count: {video_count}")
        return video_count
    else:
        print("Channel not found.")
        return None

total_videos = get_channel_video_count(API_KEY, CHANNEL_ID)

## Getting all the video IDs, titles, and the publish times using pagination:

In [0]:
def get_all_channel_videos(youtube, channel_id):
    videos = []
    next_page_token = None

    while True:
        request = youtube.search().list(
            part="snippet",
            channelId=channel_id,
            maxResults=50,
            type="video",
            order="date",
            pageToken=next_page_token
        )
        response = request.execute()

        for item in response["items"]:
            videos.append({
                "videoId": item["id"]["videoId"],
                "title": item["snippet"]["title"],
                "publishTime": item["snippet"]["publishTime"]
            })

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    print(f"Total videos retrieved: {len(videos)}")
    return videos

video_metadata = get_all_channel_videos(youtube, CHANNEL_ID)
video_metadata[:5]

## Getting the view count and the comment count for the retrieved videos:

In [0]:
def get_video_stats(youtube, video_ids):
    stats_dict = {}

    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]

        request = youtube.videos().list(
            part="statistics",
            id=",".join(batch_ids)
        )
        response = request.execute()

        for item in response["items"]:
            stats = item.get("statistics", {})
            stats_dict[item["id"]] = {
                "viewCount": int(stats.get("viewCount", 0)),
                "commentCount": int(stats.get("commentCount", 0))
            }

    return stats_dict

video_ids = [video["videoId"] for video in video_metadata]
video_stats = get_video_stats(youtube, video_ids)

list(video_stats.items())[:5]

## Combining the metadata and the statistics into a structured dataset:

In [0]:
final_data = []

for video in video_metadata:
    vid = video["videoId"]
    stats = video_stats.get(vid, {"viewCount": 0, "commentCount": 0})

    final_data.append({
        "videoId": vid,
        "title": video["title"],
        "publishTime": video["publishTime"],
        "viewCount": stats["viewCount"],
        "commentCount": stats["commentCount"]
    })

df = pd.DataFrame(final_data)

df["publishTime"] = pd.to_datetime(df["publishTime"])
df = df.sort_values("publishTime", ascending=True).reset_index(drop=True)

df.head()

## Displaying the final dataset information:

In [0]:
print("Dataset shape:", df.shape)
df.info()

## Finally, saving the dataset as a CSV file:

In [0]:
df.to_csv("yt_data_3blue1brown.csv", index=False)
print("CSV file saved successfully as yt_data_3blue1brown.csv")

# Task 2
## Converting the publish time to datetime format and analyzing the trend of views over time and plotting it on a graph:

In [0]:
import matplotlib.pyplot as plt

## Ensuring that the publishTime column is in datetime format and sorting the dataset by publish time (for the graph):

In [0]:
df["publishTime"] = pd.to_datetime(df["publishTime"])
df = df.sort_values("publishTime").reset_index(drop=True)

df[["publishTime", "viewCount"]].head()

## Plotting the trend of view count over time:

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.figure(figsize=(14, 6))
plt.plot(df["publishTime"], df["viewCount"], linewidth=2)
plt.fill_between(df["publishTime"], df["viewCount"], alpha=0.3)

plt.title("3Blue1Brown Video View Trend Over Time")
plt.xlabel("Publish Time")
plt.ylabel("View Count (Millions)")

ax = plt.gca()
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x/1_000_000:.1f}M"))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## CHECK: statistics for the view trend analysis:

In [0]:
print("Earliest video in the dataset:", df["publishTime"].min())
print("Latest video in the dataset:", df["publishTime"].max())
print("Highest view count:", df["viewCount"].max())
print("Average view count:", round(df["viewCount"].mean(), 2))

# Task 3
## Generating a word cloud of all the video titles to identify prominent topics in the 3Blue1Brown channel content:

In [0]:
%pip install wordcloud
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

## Combining all the video titles into one big text string:

In [0]:
all_titles_text = " ".join(df["title"].dropna().astype(str))
all_titles_text[:11071] 

## Creating a custom stop word list to remove common words that do not add any meaning (like filler words):

In [0]:
custom_stopwords = set(STOPWORDS)

custom_stopwords.update([
    "the", "a", "an", "of", "in", "to", "for", "and", "with", "on",
    "how", "why", "what", "from", "into", "using",
    "3blue1brown", "blue", "brown"
])

## Generating and displaying the word cloud from the video titles:

In [0]:
wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    stopwords=custom_stopwords,
    collocations=False
).generate(all_titles_text)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of 3Blue1Brown Video Titles")
plt.tight_layout()
plt.show()

## CHECK: inspecting the most frequent words in the word cloud:

In [0]:
top_words = sorted(wordcloud.words_.items(), key=lambda x: x[1], reverse=True)[:5]
top_words